### ETL
#### Importaciones

In [2]:
import pandas as pd
import numpy as np
import unicodedata
import re
import uuid
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

import warnings
warnings.filterwarnings("ignore")

#### Extracción de datos 

In [3]:
septiempre_data = pd.read_excel('./Data/09_Exportaciones_2025_Septiembre.xlsx', sheet_name='Sheet1')
agosto_data = pd.read_excel('./Data/08_Exportaciones_2025_Agosto.xlsx', sheet_name='Sheet1')
julio_data = pd.read_excel('./Data/07_Exportaciones_2025_Julio.xlsx', sheet_name='Sheet1')

##### Concat de los ultimos 3 meses, para eficiencia de las transformaciones y mostrar funcionamieto de la etl

In [19]:
df = pd.concat([septiempre_data, agosto_data, julio_data], ignore_index=True)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 353564 entries, 0 to 353563
Data columns (total 72 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   FECHA_PROCESO                  353564 non-null  int64  
 1   NUMERO_SERIE                   353564 non-null  int64  
 2   OFICINA                        353564 non-null  int64  
 3   COD_ADUANA_DESPACHO            353564 non-null  int64  
 4   ADUANA_DESPACHO                353564 non-null  object 
 5   TIPO_IDENT                     353564 non-null  int64  
 6   NIT_EXPORTADOR                 352455 non-null  float64
 7   TIPO_USUARIO                   353564 non-null  int64  
 8   COD_USUARIO                    353564 non-null  int64  
 9   CLASE_EXPORTADOR               353564 non-null  int64  
 10  COD_DPTO_EXPORTADOR            353564 non-null  int64  
 11  COD_PAIS_DESTINO_NUM           353564 non-null  int64  
 12  COD_PAIS_DESTINO_ALF          

#### Transformaciones

1. Columnas utiles para los analisis

In [20]:
cols_keep = [
    "UNIDAD_FISICA",
    "CANTIDAD_UNIDADES_FISICAS",
    "PESO_BRUTO_KGS",
    "PESO_NETO_KGS",
    "VALOR_FOB_USD",
    "VALOR_FOB_PESOS",
    "VLR_SERIE_AGREGADO_NAL_USD",
    "VALOR_SERIE_FLETES_USD",
    "VALOR_SERIE_SEGUROS_USD",
    "VLR_SERIE_OTROS_GASTOS_USD",
    "COD_MONEDA_TRANSACCION",
    "COD_MODO_TRANSPORTE",
    "COD_MODALIDAD_EXPORTACION",
    "COD_PAIS_DESTINO",
    "NIT_EXPORTADOR",
    "COD_ADUANA_DESPACHO",
    "FECHA_DECLARACION_EXPORTACION",
    "MODO_TRANSPORTE",
    "MODALIDAD_EXPORTACION",
    "PAIS_DESTINO_FINAL",
    "CIUDAD_DESTINATARIO",
    "TIPO_USUARIO",
    "COD_USUARIO",
    "CLASE_EXPORTADOR",
    "COD_DPTO_EXPORTADOR",
    "RAZON_SOCIAL_EXPORTADOR",
    "DIREC_EXPORTADOR",
    "ADUANA_DESPACHO"
]

In [21]:
df_clean = df[cols_keep].copy()
df_clean

,UNIDAD_FISICA,CANTIDAD_UNIDADES_FISICAS,PESO_BRUTO_KGS,PESO_NETO_KGS,VALOR_FOB_USD,VALOR_FOB_PESOS,VLR_SERIE_AGREGADO_NAL_USD,VALOR_SERIE_FLETES_USD,VALOR_SERIE_SEGUROS_USD,VLR_SERIE_OTROS_GASTOS_USD,...,MODALIDAD_EXPORTACION,PAIS_DESTINO_FINAL,CIUDAD_DESTINATARIO,TIPO_USUARIO,COD_USUARIO,CLASE_EXPORTADOR,COD_DPTO_EXPORTADOR,RAZON_SOCIAL_EXPORTADOR,DIREC_EXPORTADOR,ADUANA_DESPACHO
0,Unidades o artículos,24.00,0.59,0.49,4.28,1.672675e+04,0.00,0.54,0.00,0.0,...,Exportación definitiva de mercancías de fabric...,Barbados,BRIDGETOWN,36,0,2,13001,AJOVER DARNEL S A S,VIA MAMONAL KM 1 CR 56 7 C 531,Aduanas de Cartagena
1,Metro cuadrado,332.80,7211.50,7038.00,1740.55,6.790390e+06,0.00,0.00,0.00,0.0,...,Exportación definitiva de mercancías de fabric...,Barbados,CHRIST CHURCH,36,0,2,11001,ALFAGRES S.A. - EN REORGANIZACION,AC 24 95 12 BG 45 PAR INDUSTRIAL PORTOS,Aduanas de Cartagena
2,Metro cuadrado,249.60,5188.84,5064.00,1305.41,5.092783e+06,0.00,0.00,0.00,0.0,...,Exportación definitiva de mercancías de fabric...,Barbados,CHRIST CHURCH,36,0,2,11001,ALFAGRES S.A. - EN REORGANIZACION,AC 24 95 12 BG 45 PAR INDUSTRIAL PORTOS,Aduanas de Cartagena
3,Metro cuadrado,238.08,3959.25,3864.00,852.33,3.325186e+06,0.00,0.00,0.00,0.0,...,Exportación definitiva de mercancías de fabric...,Barbados,CHRIST CHURCH,36,0,2,11001,ALFAGRES S.A. - EN REORGANIZACION,AC 24 95 12 BG 45 PAR INDUSTRIAL PORTOS,Aduanas de Cartagena
4,Metro cuadrado,238.08,3946.96,3852.00,852.33,3.325186e+06,0.00,0.00,0.00,0.0,...,Exportación definitiva de mercancías de fabric...,Barbados,CHRIST CHURCH,36,0,2,11001,ALFAGRES S.A. - EN REORGANIZACION,AC 24 95 12 BG 45 PAR INDUSTRIAL PORTOS,Aduanas de Cartagena
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353559,Metro cúbico,21.01,16941.31,16941.31,12506.43,5.007450e+07,0.00,0.00,0.00,0.0,...,Exportación definitiva de mercancías de fabric...,Venezuela (República Bolivariana de),BARRANQUILLA,36,0,2,11001,TERPEL EXPORTACIONES C.I S.A.S,CR 7 75 51,Aduanas de Bogotá - Aeropuerto El Dorado
353560,Kilogramo,864.00,1048.91,864.00,3000.00,1.218408e+07,2744.77,0.00,0.00,0.0,...,Exportación definitiva de mercancías de fabric...,Venezuela (República Bolivariana de),BARQUISIMETO,36,0,2,13001,PRODUCTORA DE CONFITES Y CHICLET S MAC DULCES SAS,BRR BOSQUE TV 54 21 A 120 CENTRO EMPR...,Aduanas de Cúcuta
353561,Kilogramo,5262.00,5262.00,5262.00,3367.68,1.351618e+07,0.00,0.00,0.00,0.0,...,Las demás exportaciones definitivas no incluid...,Venezuela (República Bolivariana de),BARQUISIMETO,36,0,2,5001,CARTON DE COLOMBIA S.A.,CR 64 67 B 35 BL 3 P 2,Aduanas de Cúcuta
353562,Kilogramo,6346.20,7961.40,6346.20,16904.11,6.853146e+07,16643.86,2154.04,0.65,0.0,...,Exportación definitiva de mercancías de fabric...,Venezuela (República Bolivariana de),CARACAS,36,0,2,76001,COLGATE PALMOLIVE COMPAÑIA,CL 41 NORTE 4 N 11,Aduanas de Cúcuta


2. tratamiento del NIT

In [25]:
df_clean["NIT_EXPORTADOR"] = df_clean["NIT_EXPORTADOR"].fillna(0)
df_clean.loc[df_clean["NIT_EXPORTADOR"] == 0, "NIT_EXPORTADOR"] = 9999999999

df_clean[df_clean["NIT_EXPORTADOR"] == 9999999999]["NIT_EXPORTADOR"].value_counts()

NIT_EXPORTADOR
1.000000e+10    1823
Name: count, dtype: int64

3. Tratamiendo de las ciudades

In [22]:
def limpiar_ciudad(nombre):
    nombre = str(nombre).upper().strip()

    # Eliminar acentos y caracteres especiales
    nombre = ''.join(
        c for c in unicodedata.normalize('NFD', nombre)
        if unicodedata.category(c) != 'Mn'
    )

    # Quitar codificaciones rotas tipo &#65533
    nombre = re.sub(r'&#\d+;?', '', nombre)

    # Eliminar paréntesis y texto dentro
    nombre = re.sub(r'\(.*?\)', '', nombre)

    # Eliminar comas, puntos, y sufijos administrativos como DC o D.C.
    nombre = re.sub(r',?\s*(D\.?C\.?|DC)$', '', nombre)
    nombre = re.sub(r'[^A-Z\s]', '', nombre)
    
    # Limpiar espacios repetidos
    nombre = re.sub(r'\s+', ' ', nombre).strip()

    # Homologar equivalencias comunes
    equivalencias = {
        "BOGOT": "BOGOTA",
        "BOGOTA DC": "BOGOTA",
        "SANTA FE DE BOGOTA": "BOGOTA",
        "MEDELLIN": "MEDELLIN",
        "MEDELLN": "MEDELLIN",
        "NUNCHIA": "NUNCHIA",
        "COTA": "COTA",
        "FUNZA": "FUNZA",
        "SANTA MARTA": "SANTA MARTA",
        "CARTAGENA": "CARTAGENA",
        "PEREIRA": "PEREIRA",
        "BARRANQUILLA": "BARRANQUILLA",
        "MOSQUERA": "MOSQUERA",
        "MACEIO": "MACEO",
    }

    # Reemplazo por equivalencia si existe
    if nombre in equivalencias:
        nombre = equivalencias[nombre]

    return nombre

In [23]:

df_clean["CIUDAD_DESTINATARIO"] = df_clean["CIUDAD_DESTINATARIO"].fillna("DESCONOCIDO")
df_clean["CIUDAD_DESTINATARIO"] = df_clean["CIUDAD_DESTINATARIO"].apply(limpiar_ciudad)
df_clean.loc[df_clean["COD_PAIS_DESTINO"] == "XCF", ["COD_PAIS_DESTINO", "PAIS_DESTINO_FINAL"]] = ["COL", "Colombia"]
df_clean[df_clean["COD_PAIS_DESTINO"] == "COL"]["CIUDAD_DESTINATARIO"].unique()

array(['NUNCHA', 'SONSON', 'BOGOTA', 'MEDELLIN', 'RIONEGRO',
       'DESCONOCIDO', 'SANTA MARTA', 'PALMIRA', 'BARRANQUILLA',
       'VILLA RICA', 'TURBACO', 'PEREIRA', 'PUERTO TEJADA', 'COTA',
       'FUNZA', 'CARTAGENA', 'PUERTO LOPEZ', 'C'], dtype=object)

In [47]:
df_clean["ID_DESTINO"] = df_clean["COD_PAIS_DESTINO"] + "_" + df_clean["CIUDAD_DESTINATARIO"]
df_clean["ID_DESTINO"] = df_clean["ID_DESTINO"].str.replace(" ", "_")
df_clean["ID_DESTINO"] = df_clean["ID_DESTINO"].str.replace("-", "_")
df_clean["ID_DESTINO"] = df_clean["ID_DESTINO"].str.replace(".", "_")
df_clean["ID_DESTINO"] = df_clean["ID_DESTINO"].str.upper()
df_clean["ID_DESTINO"].value_counts()

ID_DESTINO
USA_MIAMI               29186
ECU_QUITO               28467
PER_LIMA                21363
CRI_SAN_JOSE            11887
ECU_GUAYAQUIL            8593
                        ...  
PRT_CANELAS                 1
NLD_GELDERLAND_VUREN        1
DNK_AARHUS                  1
CZE_PRAGUE                  1
USA_MOSINEE                 1
Name: count, Length: 6533, dtype: int64

4. Tratamiento para las fechas

In [24]:
df_clean["FECHA_DECLARACION_EXPORTACION"] = pd.to_datetime(df_clean["FECHA_DECLARACION_EXPORTACION"], format="%Y%m%d")
df_clean["AÑO"] = df_clean["FECHA_DECLARACION_EXPORTACION"].dt.year
df_clean["MES"] = df_clean["FECHA_DECLARACION_EXPORTACION"].dt.month
df_clean["DIA"] = df_clean["FECHA_DECLARACION_EXPORTACION"].dt.day
df_clean

,UNIDAD_FISICA,CANTIDAD_UNIDADES_FISICAS,PESO_BRUTO_KGS,PESO_NETO_KGS,VALOR_FOB_USD,VALOR_FOB_PESOS,VLR_SERIE_AGREGADO_NAL_USD,VALOR_SERIE_FLETES_USD,VALOR_SERIE_SEGUROS_USD,VLR_SERIE_OTROS_GASTOS_USD,...,TIPO_USUARIO,COD_USUARIO,CLASE_EXPORTADOR,COD_DPTO_EXPORTADOR,RAZON_SOCIAL_EXPORTADOR,DIREC_EXPORTADOR,ADUANA_DESPACHO,AÑO,MES,DIA
0,Unidades o artículos,24.00,0.59,0.49,4.28,1.672675e+04,0.00,0.54,0.00,0.0,...,36,0,2,13001,AJOVER DARNEL S A S,VIA MAMONAL KM 1 CR 56 7 C 531,Aduanas de Cartagena,2025,9,29
1,Metro cuadrado,332.80,7211.50,7038.00,1740.55,6.790390e+06,0.00,0.00,0.00,0.0,...,36,0,2,11001,ALFAGRES S.A. - EN REORGANIZACION,AC 24 95 12 BG 45 PAR INDUSTRIAL PORTOS,Aduanas de Cartagena,2025,9,30
2,Metro cuadrado,249.60,5188.84,5064.00,1305.41,5.092783e+06,0.00,0.00,0.00,0.0,...,36,0,2,11001,ALFAGRES S.A. - EN REORGANIZACION,AC 24 95 12 BG 45 PAR INDUSTRIAL PORTOS,Aduanas de Cartagena,2025,9,30
3,Metro cuadrado,238.08,3959.25,3864.00,852.33,3.325186e+06,0.00,0.00,0.00,0.0,...,36,0,2,11001,ALFAGRES S.A. - EN REORGANIZACION,AC 24 95 12 BG 45 PAR INDUSTRIAL PORTOS,Aduanas de Cartagena,2025,9,30
4,Metro cuadrado,238.08,3946.96,3852.00,852.33,3.325186e+06,0.00,0.00,0.00,0.0,...,36,0,2,11001,ALFAGRES S.A. - EN REORGANIZACION,AC 24 95 12 BG 45 PAR INDUSTRIAL PORTOS,Aduanas de Cartagena,2025,9,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353559,Metro cúbico,21.01,16941.31,16941.31,12506.43,5.007450e+07,0.00,0.00,0.00,0.0,...,36,0,2,11001,TERPEL EXPORTACIONES C.I S.A.S,CR 7 75 51,Aduanas de Bogotá - Aeropuerto El Dorado,2025,7,13
353560,Kilogramo,864.00,1048.91,864.00,3000.00,1.218408e+07,2744.77,0.00,0.00,0.0,...,36,0,2,13001,PRODUCTORA DE CONFITES Y CHICLET S MAC DULCES SAS,BRR BOSQUE TV 54 21 A 120 CENTRO EMPR...,Aduanas de Cúcuta,2025,7,23
353561,Kilogramo,5262.00,5262.00,5262.00,3367.68,1.351618e+07,0.00,0.00,0.00,0.0,...,36,0,2,5001,CARTON DE COLOMBIA S.A.,CR 64 67 B 35 BL 3 P 2,Aduanas de Cúcuta,2025,7,11
353562,Kilogramo,6346.20,7961.40,6346.20,16904.11,6.853146e+07,16643.86,2154.04,0.65,0.0,...,36,0,2,76001,COLGATE PALMOLIVE COMPAÑIA,CL 41 NORTE 4 N 11,Aduanas de Cúcuta,2025,7,9


In [26]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 353564 entries, 0 to 353563
Data columns (total 31 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   UNIDAD_FISICA                  353564 non-null  object        
 1   CANTIDAD_UNIDADES_FISICAS      353564 non-null  float64       
 2   PESO_BRUTO_KGS                 353564 non-null  float64       
 3   PESO_NETO_KGS                  353564 non-null  float64       
 4   VALOR_FOB_USD                  353564 non-null  float64       
 5   VALOR_FOB_PESOS                353564 non-null  float64       
 6   VLR_SERIE_AGREGADO_NAL_USD     353564 non-null  float64       
 7   VALOR_SERIE_FLETES_USD         353564 non-null  float64       
 8   VALOR_SERIE_SEGUROS_USD        353564 non-null  float64       
 9   VLR_SERIE_OTROS_GASTOS_USD     353564 non-null  float64       
 10  COD_MONEDA_TRANSACCION         353564 non-null  object        
 11  

#### LOAD
Creción de las dimensiones y carga a la BD de postgres

In [40]:
dim_time = df_clean[["FECHA_DECLARACION_EXPORTACION"]].drop_duplicates().copy()
dim_time["FECHA_DECLARACION_EXPORTACION"] = pd.to_datetime(dim_time["FECHA_DECLARACION_EXPORTACION"], format="%Y%m%d")
dim_time["anio"] = dim_time["FECHA_DECLARACION_EXPORTACION"].dt.year
dim_time["mes"] = dim_time["FECHA_DECLARACION_EXPORTACION"].dt.month
dim_time["dia"] = dim_time["FECHA_DECLARACION_EXPORTACION"].dt.day
dim_time = dim_time[["FECHA_DECLARACION_EXPORTACION", "anio", "mes", "dia"]]
dim_time.sort_values("FECHA_DECLARACION_EXPORTACION", inplace=True)
dim_time = dim_time.rename(columns={"FECHA_DECLARACION_EXPORTACION": "id_tiempo"})
dim_time = dim_time.reset_index(drop=True)
dim_time

,id_tiempo,anio,mes,dia
0,2025-07-01,2025,7,1
1,2025-07-02,2025,7,2
2,2025-07-03,2025,7,3
3,2025-07-04,2025,7,4
4,2025-07-05,2025,7,5
...,...,...,...,...
87,2025-09-26,2025,9,26
88,2025-09-27,2025,9,27
89,2025-09-28,2025,9,28
90,2025-09-29,2025,9,29


In [41]:
dim_empresa = df_clean[["NIT_EXPORTADOR", "TIPO_USUARIO", "COD_USUARIO", "CLASE_EXPORTADOR", "COD_DPTO_EXPORTADOR", "RAZON_SOCIAL_EXPORTADOR", "DIREC_EXPORTADOR"]].drop_duplicates().copy()
dim_empresa = dim_empresa.reset_index(drop=True)
dim_empresa = dim_empresa.rename(columns={"NIT_EXPORTADOR": "nit", "TIPO_USUARIO": "tipo_usuario", "COD_USUARIO": "cod_usuario", "CLASE_EXPORTADOR": "clase", "COD_DPTO_EXPORTADOR": "cod_departamento", "RAZON_SOCIAL_EXPORTADOR": "razon_social", "DIREC_EXPORTADOR": "direccion"})
dim_empresa = dim_empresa.drop_duplicates(subset=["nit"], keep="first").copy()
dim_empresa["nit"] = dim_empresa["nit"].astype(float).astype("Int64").astype(str)
dim_empresa

,nit,tipo_usuario,cod_usuario,clase,cod_departamento,razon_social,direccion
0,860013771,36,0,2,13001,AJOVER DARNEL S A S,VIA MAMONAL KM 1 CR 56 7 C 531
1,860032550,36,0,2,11001,ALFAGRES S.A. - EN REORGANIZACION,AC 24 95 12 BG 45 PAR INDUSTRIAL PORTOS
2,860002536,36,0,2,5001,COMPAÑIA COLOMBIANA DE CERAMICA S A S,CL 16 41 210 OF 103 104 406 ED LA ...
3,830130106,36,0,2,11001,SOENERGY INTERNATIONAL COLOMBIA S.A.S.,AC 24 95 12 BG 45 PAR INDUSTRIAL PORTOS
4,830021253,36,0,2,11001,CROWN COLOMBIANA S.A.,CL 25 G 100 24
...,...,...,...,...,...,...,...
8556,900864818,36,0,2,5266,INTERLUBGROUP COLOMBIA S.A.S.,CL 23 SUR 42 B 60 OF 201 ED O'CLOCK
8558,860528320,36,0,2,54001,MICROLINK S A S,AV LIBERTADORES 15 N 119 BL C LC 25 Y 26 CO...
8563,805006906,36,0,2,54001,REPRESENTACIONES INDUSTRIALES DE COLOMBIA RICO...,CL 9 N 2 06 URB EL BOSQUE
8564,860066323,36,0,2,54001,POLIETILENOS DEL VALLE S.A,ZF CUCUTA BG D 3 1


In [42]:
dim_transporte = df_clean[["COD_MODO_TRANSPORTE", "MODO_TRANSPORTE"]].drop_duplicates().copy()
dim_transporte = dim_transporte.reset_index(drop=True)
dim_transporte = dim_transporte.rename(columns={"COD_MODO_TRANSPORTE": "cod_modo_transporte", "MODO_TRANSPORTE": "modo_transporte"})
dim_transporte

,cod_modo_transporte,modo_transporte
0,1,Marítimo
1,7,Instalaciones Fijas
2,4,Aéreo
3,3,Terrestre (carretero)
4,8,Vías navegables interiores


In [43]:
dim_aduanas = df_clean[["COD_ADUANA_DESPACHO", "ADUANA_DESPACHO"]].drop_duplicates().copy()
dim_aduanas = dim_aduanas.reset_index(drop=True)
dim_aduanas = dim_aduanas.rename(columns={"COD_ADUANA_DESPACHO": "cod_aduana_despacho", "ADUANA_DESPACHO": "aduana_despacho"})
dim_aduanas

,cod_aduana_despacho,aduana_despacho
0,48,Aduanas de Cartagena
1,87,Aduanas de Barranquilla
2,19,Impuestos y Aduanas de Santa Marta
3,90,Aduanas de Medellín
4,3,Aduanas de Bogotá - Aeropuerto El Dorado
5,41,Impuestos y Aduanas de Urabá
6,35,Impuestos y Aduanas de Buenaventura
7,25,Impuestos y Aduanas de Riohacha
8,88,Aduanas de Cali
9,89,Aduanas de Cúcuta


In [44]:
worldcities = pd.read_csv("./Data/worldcities.csv")
worldcities_subset = worldcities[["city", "lat", "lng", "iso3"]].copy()
worldcities_subset["city"] = worldcities_subset["city"].str.upper().str.strip()
worldcities_subset["iso3"] = worldcities_subset["iso3"].str.upper().str.strip()

world_latlng_by_country = worldcities_subset.groupby("iso3")[["lat", "lng"]].mean().reset_index().rename(columns={"iso3": "COD_PAIS_DESTINO", "lat": "LATITUD", "lng": "LONGITUD"})

In [48]:
dim_destino = df_clean[["ID_DESTINO", "COD_PAIS_DESTINO", "PAIS_DESTINO_FINAL", "CIUDAD_DESTINATARIO"]].drop_duplicates().copy()
dim_destino = dim_destino.reset_index(drop=True)
dim_destino_geo = dim_destino.merge(world_latlng_by_country, on="COD_PAIS_DESTINO", how="left")
dim_destino_geo.isnull().sum()

ID_DESTINO             0
COD_PAIS_DESTINO       0
PAIS_DESTINO_FINAL     0
CIUDAD_DESTINATARIO    0
LATITUD                1
LONGITUD               1
dtype: int64

In [50]:
faltantes_geo = dim_destino[dim_destino_geo["LATITUD"].isna() | dim_destino_geo["LONGITUD"].isna()]
faltantes_geo

,ID_DESTINO,COD_PAIS_DESTINO,PAIS_DESTINO_FINAL,CIUDAD_DESTINATARIO
1264,PSE_HEBRON,PSE,Estado de Palestina,HEBRON


In [51]:
# Hebrón (PSE)
mask_pse = (dim_destino_geo["COD_PAIS_DESTINO"]=="PSE") & (dim_destino_geo["CIUDAD_DESTINATARIO"].str.upper().str.contains("HEBRON"))
dim_destino_geo.loc[mask_pse, ["LATITUD","LONGITUD"]] = [31.5294, 35.0938]

# Providenciales (IOT)
mask_iot = (dim_destino_geo["COD_PAIS_DESTINO"]=="IOT") & (dim_destino_geo["CIUDAD_DESTINATARIO"].str.upper().str.contains("PROVIDENCIALES"))
dim_destino_geo.loc[mask_iot, ["LATITUD","LONGITUD"]] = [-7.3195, 72.422859]

dim_destino_geo.isnull().sum()

ID_DESTINO             0
COD_PAIS_DESTINO       0
PAIS_DESTINO_FINAL     0
CIUDAD_DESTINATARIO    0
LATITUD                0
LONGITUD               0
dtype: int64

In [52]:
dim_destino_geo = dim_destino_geo.rename(columns={"ID_DESTINO": "id_destino", "COD_PAIS_DESTINO": "cod_pais_destino", "PAIS_DESTINO_FINAL": "pais_destino_final", "CIUDAD_DESTINATARIO": "ciudad_destinatario", "LATITUD": "latitud", "LONGITUD": "longitud"})
dim_destino_geo

,id_destino,cod_pais_destino,pais_destino_final,ciudad_destinatario,latitud,longitud
0,BRB_BRIDGETOWN,BRB,Barbados,BRIDGETOWN,13.096900,-59.613100
1,BRB_CHRIST_CHURCH,BRB,Barbados,CHRIST CHURCH,13.096900,-59.613100
2,BRB_BARBADOS,BRB,Barbados,BARBADOS,13.096900,-59.613100
3,BHS_GRAND_BAHAMA,BHS,Bahamas,GRAND BAHAMA,26.192800,-78.416275
4,BRB_NEWTON,BRB,Barbados,NEWTON,13.096900,-59.613100
...,...,...,...,...,...,...
6528,VEN_FORT_LAUDERDALE,VEN,Venezuela (República Bolivariana de),FORT LAUDERDALE,9.574322,-66.576424
6529,VEN_SAN_CRISTOABL,VEN,Venezuela (República Bolivariana de),SAN CRISTOABL,9.574322,-66.576424
6530,VEN_ST_BARBARA_DEL_ZULIA,VEN,Venezuela (República Bolivariana de),ST BARBARA DEL ZULIA,9.574322,-66.576424
6531,CRI_CUMANA,CRI,Costa Rica,CUMANA,10.002173,-84.137714


In [54]:
dim_modalidad = df_clean[["COD_MODALIDAD_EXPORTACION", "MODALIDAD_EXPORTACION"]].drop_duplicates().copy()
dim_modalidad = dim_modalidad.reset_index(drop=True)
dim_modalidad = dim_modalidad.rename(columns={"COD_MODALIDAD_EXPORTACION": "cod_modalidad_exportacion", "MODALIDAD_EXPORTACION": "modalidad_exportacion"})
dim_modalidad

,cod_modalidad_exportacion,modalidad_exportacion
0,198,Exportación definitiva de mercancías de fabric...
1,104,Exportación definitiva de mercancías que resul...
2,107,Donaciones
3,2,Muestras sin valor comercial
4,199,Las demás exportaciones definitivas no incluid...
5,4,Menaje
6,401,Reexportación definitiva de mercancías que est...
7,201,Mercancías exportadas temporalmente para trans...
8,310,Exportación de mercancías en consignación.
9,402,Reexportación definitiva de mercancías import...


In [55]:
fact_exportaciones = df_clean[[
    "UNIDAD_FISICA",
    "CANTIDAD_UNIDADES_FISICAS",
    "PESO_BRUTO_KGS",
    "PESO_NETO_KGS",
    "VALOR_FOB_USD",
    "VALOR_FOB_PESOS",
    "VLR_SERIE_AGREGADO_NAL_USD",
    "VALOR_SERIE_FLETES_USD",
    "VALOR_SERIE_SEGUROS_USD",
    "VLR_SERIE_OTROS_GASTOS_USD",
    "COD_MONEDA_TRANSACCION",
    "COD_MODO_TRANSPORTE",
    "COD_MODALIDAD_EXPORTACION",
    "NIT_EXPORTADOR",
    "COD_ADUANA_DESPACHO",
    "FECHA_DECLARACION_EXPORTACION",
    "ID_DESTINO"
]].copy()

In [56]:
fact_exportaciones["ID_EXPORTACION"] = [uuid.uuid4().hex for _ in range(len(fact_exportaciones))]


In [57]:
fact_exportaciones = fact_exportaciones[[
    "ID_EXPORTACION",
    "NIT_EXPORTADOR",
    "COD_ADUANA_DESPACHO",
    "COD_MODO_TRANSPORTE",
    "COD_MODALIDAD_EXPORTACION",
    "FECHA_DECLARACION_EXPORTACION",
    "ID_DESTINO",
    "UNIDAD_FISICA",
    "CANTIDAD_UNIDADES_FISICAS",
    "PESO_BRUTO_KGS",
    "PESO_NETO_KGS",
    "VALOR_FOB_USD",
    "VALOR_FOB_PESOS",
    "VLR_SERIE_AGREGADO_NAL_USD",
    "VALOR_SERIE_FLETES_USD",
    "VALOR_SERIE_SEGUROS_USD",
    "VLR_SERIE_OTROS_GASTOS_USD",
    "COD_MONEDA_TRANSACCION"
]]
fact_exportaciones = fact_exportaciones.rename(columns={
    "ID_EXPORTACION": "id_exportacion",
    "NIT_EXPORTADOR": "nit",
    "COD_ADUANA_DESPACHO": "cod_aduana_despacho",
    "COD_MODO_TRANSPORTE": "cod_modo_transporte",
    "COD_MODALIDAD_EXPORTACION": "cod_modalidad_exportacion",
    "FECHA_DECLARACION_EXPORTACION": "id_tiempo",
    "ID_DESTINO": "id_destino",
    "UNIDAD_FISICA": "unidad_fisica",
    "CANTIDAD_UNIDADES_FISICAS": "cantidad_unidades_fisicas",
    "PESO_BRUTO_KGS": "peso_bruto_kgs",
    "PESO_NETO_KGS": "peso_neto_kgs",
    "VALOR_FOB_USD": "valor_fob_usd",
    "VALOR_FOB_PESOS": "valor_fob_pesos",
    "VLR_SERIE_AGREGADO_NAL_USD": "vlr_serie_agregado_nal_usd",
    "VALOR_SERIE_FLETES_USD": "valor_serie_fletes_usd",
    "VALOR_SERIE_SEGUROS_USD": "valor_serie_seguros_usd",
    "VLR_SERIE_OTROS_GASTOS_USD": "valor_serie_otros_gastos_usd",
    "COD_MONEDA_TRANSACCION": "moneda_transaccion"
})
fact_exportaciones["nit"] = fact_exportaciones["nit"].astype(float).astype("Int64").astype(str)
fact_exportaciones

,id_exportacion,nit,cod_aduana_despacho,cod_modo_transporte,cod_modalidad_exportacion,id_tiempo,id_destino,unidad_fisica,cantidad_unidades_fisicas,peso_bruto_kgs,peso_neto_kgs,valor_fob_usd,valor_fob_pesos,vlr_serie_agregado_nal_usd,valor_serie_fletes_usd,valor_serie_seguros_usd,valor_serie_otros_gastos_usd,moneda_transaccion
0,d35881051dc04c099cc73dcbe24ee944,860013771,48,1,198,2025-09-29,BRB_BRIDGETOWN,Unidades o artículos,24.00,0.59,0.49,4.28,1.672675e+04,0.00,0.54,0.00,0.0,USD
1,5e7f31df197443f0b7b105dbf7fcfde5,860032550,48,1,198,2025-09-30,BRB_CHRIST_CHURCH,Metro cuadrado,332.80,7211.50,7038.00,1740.55,6.790390e+06,0.00,0.00,0.00,0.0,USD
2,b828c7bf9d2040cdb3a0d4a543d261ae,860032550,48,1,198,2025-09-30,BRB_CHRIST_CHURCH,Metro cuadrado,249.60,5188.84,5064.00,1305.41,5.092783e+06,0.00,0.00,0.00,0.0,USD
3,2fbb36508d5d4aa596c674d21ebf4a77,860032550,48,1,198,2025-09-30,BRB_CHRIST_CHURCH,Metro cuadrado,238.08,3959.25,3864.00,852.33,3.325186e+06,0.00,0.00,0.00,0.0,USD
4,b7946b5e47ad48e7ac953a6adfd1d71c,860032550,48,1,198,2025-09-30,BRB_CHRIST_CHURCH,Metro cuadrado,238.08,3946.96,3852.00,852.33,3.325186e+06,0.00,0.00,0.00,0.0,USD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353559,1a63639a76b74d08abab252889a0e2da,901210452,3,7,198,2025-07-13,VEN_BARRANQUILLA,Metro cúbico,21.01,16941.31,16941.31,12506.43,5.007450e+07,0.00,0.00,0.00,0.0,COP
353560,116cbd1d2593420a887726ec3b0173fd,805027332,89,3,198,2025-07-23,VEN_BARQUISIMETO,Kilogramo,864.00,1048.91,864.00,3000.00,1.218408e+07,2744.77,0.00,0.00,0.0,USD
353561,38a1f71a352f41e1a4e3a0991bedc306,890300406,89,3,199,2025-07-11,VEN_BARQUISIMETO,Kilogramo,5262.00,5262.00,5262.00,3367.68,1.351618e+07,0.00,0.00,0.00,0.0,USD
353562,694ff5c59ea84038b522a0c81b05013f,890300546,89,3,198,2025-07-09,VEN_CARACAS,Kilogramo,6346.20,7961.40,6346.20,16904.11,6.853146e+07,16643.86,2154.04,0.65,0.0,COP


In [59]:
import psycopg2
from sqlalchemy import create_engine

USER = "postgres"
PASSWORD = "Psa12345"
HOST = "localhost"
PORT = "5432"
DB = "expo"

engine = create_engine(f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}")
conn = engine.connect()

In [60]:
dim_time.to_sql("dim_time", engine, if_exists="append", index=False)
dim_empresa.to_sql("dim_empresa", engine, if_exists="append", index=False)
dim_transporte.to_sql("dim_transporte", engine, if_exists="append", index=False)
dim_aduanas.to_sql("dim_aduanas", engine, if_exists="append", index=False)
dim_destino_geo.to_sql("dim_destino", engine, if_exists="append", index=False)
dim_modalidad.to_sql("dim_modalidad", engine, if_exists="append", index=False)
fact_exportaciones.to_sql("fact_exportaciones", engine, if_exists="append", index=False, chunksize=50000)

7564

#### CONSULTAS 

1. Top empresas ultimo mes 

In [ ]:

def run_sql(sql):
    return pd.read_sql(sql, engine)

top_empresas = run_sql("""
WITH last_month AS (
  SELECT DATE '2025-09-01' AS month_start
)
SELECT
  e.razon_social,
  f.nit,
  SUM(f.valor_fob_usd) AS total_fob_usd,
  SUM(f.valor_fob_pesos) AS total_fob_pesos
FROM fact_exportaciones f
JOIN dim_time t ON f.id_tiempo = t.id_tiempo
JOIN dim_empresa e ON f.nit = e.nit
WHERE date_trunc('month', t.id_tiempo) = (SELECT month_start FROM last_month)
GROUP BY e.razon_social, f.nit
ORDER BY total_fob_usd DESC
LIMIT 20;
""")


print(top_empresas.head(10))


                                      razon_social        nit  total_fob_usd  \
0                                    ECOPETROL S A  899999068   7.077293e+08   
1       CONTINENTAL GOLD LIMITED SUCURSAL COLOMBIA  900166687   3.267291e+08   
2                                     DRUMMOND LTD  800021308   2.308423e+08   
3                              ARIS MINING SEGOVIA  900306309   9.999786e+07   
4                    CARBONES DEL CERREJON LIMITED  860069804   8.917763e+07   
5         COMERCIALIZADORA INTERNACIONAL ESLOP SAS  901019586   7.611668e+07   
6            LOUIS DREYFUS COMPANY COLOMBIA S.A.S.  900174478   7.127542e+07   
7         C.I. TRAFIGURA PETROLEUM COLOMBIA S.A.S.  900585067   6.480403e+07   
8  FRONTERA ENERGY COLOMBIA CORP SUCURSAL COLOMBIA  830126302   6.290470e+07   
9                   TERPEL EXPORTACIONES C.I S.A.S  901210452   5.480286e+07   

   total_fob_pesos  
0     2.780014e+12  
1     1.304100e+12  
2     9.064935e+11  
3     3.924889e+11  
4     3.477104

2. precio total del los ultimos meses 

In [65]:
serie_mensual = run_sql("""
SELECT
  date_trunc('month', id_tiempo)::date AS month_start,
  SUM(valor_fob_usd) AS total_fob_usd
FROM fact_exportaciones
GROUP BY 1
ORDER BY total_fob_usd DESC;
""")

print(serie_mensual.head(10))

  month_start  total_fob_usd
0  2025-09-01   4.621300e+09
1  2025-07-01   4.427485e+09
2  2025-08-01   3.871217e+09


3. Destinos a los que se ha enviado más en los ultimos 6 meses ordenados por num shipments

In [66]:
top_destinos_6m = run_sql("""
WITH max_month AS (
  SELECT date_trunc('month', MAX(id_tiempo))::date AS max_month
  FROM dim_time
), window_period AS (
  SELECT (max_month - interval '5 months')::date AS start_month, max_month FROM max_month
)
SELECT
  d.id_destino,
  d.pais_destino_final,
  d.ciudad_destinatario,
  SUM(f.valor_fob_usd) AS total_fob_usd,
  COUNT(*) AS num_shipments
FROM fact_exportaciones f
JOIN dim_time t ON f.id_tiempo = t.id_tiempo
JOIN dim_destino d ON f.id_destino = d.id_destino
WHERE date_trunc('month', t.id_tiempo)::date BETWEEN (SELECT start_month FROM window_period) AND (SELECT max_month FROM window_period)
GROUP BY d.id_destino, d.pais_destino_final, d.ciudad_destinatario
ORDER BY num_shipments DESC
LIMIT 20;
""")
print(top_destinos_6m.head(20))

                 id_destino                    pais_destino_final  \
0                 USA_MIAMI             Estados Unidos de América   
1                 ECU_QUITO                               Ecuador   
2                  PER_LIMA                                  Perú   
3              CRI_SAN_JOSE                            Costa Rica   
4             ECU_GUAYAQUIL                               Ecuador   
5         DOM_SANTO_DOMINGO                  República Dominicana   
6      PAN_CIUDAD_DE_PANAMA                                Panamá   
7     CHL_SANTIAGO_DE_CHILE                                 Chile   
8             GTM_GUATEMALA                             Guatemala   
9      MEX_CIUDAD_DE_MEXICO                                México   
10              VEN_CARACAS  Venezuela (República Bolivariana de)   
11  GTM_CIUDAD_DE_GUATEMALA                             Guatemala   
12             CHL_SANTIAGO                                 Chile   
13               PAN_PANAMA       

4. Top 5 exportadores por cada tipo de tansporte 

In [71]:
top_exportadores_por_transporte = run_sql("""
WITH last_month AS (
  SELECT date_trunc('month', MAX(id_tiempo))::date AS month_start
  FROM dim_time
), agg AS (
  SELECT
    f.cod_modo_transporte,
    dt.modo_transporte,
    f.nit,
    e.razon_social,
    SUM(f.valor_fob_usd) AS total_fob_usd,
    COUNT(*) AS num_shipments
  FROM fact_exportaciones f
  JOIN dim_time t ON f.id_tiempo = t.id_tiempo
  LEFT JOIN dim_transporte dt ON f.cod_modo_transporte = dt.cod_modo_transporte
  LEFT JOIN dim_empresa e ON f.nit = e.nit
  WHERE date_trunc('month', t.id_tiempo) = (SELECT month_start FROM last_month)
  GROUP BY f.cod_modo_transporte, dt.modo_transporte, f.nit, e.razon_social
)
SELECT
  cod_modo_transporte,
  modo_transporte,
  nit,
  razon_social,
  total_fob_usd,
  num_shipments
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY cod_modo_transporte ORDER BY total_fob_usd DESC) AS rn
  FROM agg
) t
WHERE rn <= 5
ORDER BY cod_modo_transporte, num_shipments DESC;
""")

print(top_exportadores_por_transporte)

    cod_modo_transporte             modo_transporte        nit  \
0                     1                    Marítimo  800021308   
1                     1                    Marítimo  900174478   
2                     1                    Marítimo  899999068   
3                     1                    Marítimo  860069804   
4                     1                    Marítimo  900166687   
5                     3       Terrestre (carretero)  860025792   
6                     3       Terrestre (carretero)  890300431   
7                     3       Terrestre (carretero)  901597862   
8                     3       Terrestre (carretero)  819004712   
9                     3       Terrestre (carretero)  890700058   
10                    4                       Aéreo  901019586   
11                    4                       Aéreo  890902070   
12                    4                       Aéreo  900306309   
13                    4                       Aéreo  901218630   
14        

5. Resumen del ultimo mes de exportaciones

In [72]:
resumen_transporte_ultimo_mes = run_sql("""
WITH last_month AS (
  SELECT date_trunc('month', MAX(id_tiempo))::date AS month_start FROM dim_time
)
SELECT
  f.cod_modo_transporte,
  dt.modo_transporte,
  COUNT(*) AS num_shipments,
  SUM(f.valor_fob_usd) AS total_fob_usd,
  AVG(f.valor_fob_usd) AS avg_fob_usd
FROM fact_exportaciones f
JOIN dim_time t ON f.id_tiempo = t.id_tiempo
LEFT JOIN dim_transporte dt ON f.cod_modo_transporte = dt.cod_modo_transporte
WHERE date_trunc('month', t.id_tiempo) = (SELECT month_start FROM last_month)
GROUP BY f.cod_modo_transporte, dt.modo_transporte
ORDER BY total_fob_usd DESC;
""")
print(resumen_transporte_ultimo_mes)

   cod_modo_transporte             modo_transporte  num_shipments  \
0                    1                    Marítimo          51930   
1                    4                       Aéreo          48337   
2                    3       Terrestre (carretero)          20490   
3                    7         Instalaciones Fijas            667   
4                    8  Vías navegables interiores              2   

   total_fob_usd    avg_fob_usd  
0   3.578762e+09   68915.123652  
1   6.935274e+08   14347.753491  
2   2.808922e+08   13708.746944  
3   6.811181e+07  102116.651724  
4   5.983200e+03    2991.600000  
